<a href="https://colab.research.google.com/github/lcqsigi/big-data2/blob/main/nvidia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install nvalchemi-toolkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 332.9/332.9 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 811.1/811.1 kB 55.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 464.2/464.2 kB 42.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.8/536.8 kB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.6/319.6 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 90.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.3/138.3 MB 7.3 MB/s eta 0:00:00


In [19]:
pip install ase

In [4]:
from __future__ import annotations

from pathlib import Path

import torch
from ase import Atoms
from ase.build import fcc111, molecule
from ase.io import write

from nvalchemi._typing import AtomCategory
from nvalchemi.data import AtomicData, Batch
from nvalchemi.dynamics import FIRE, NVTLangevin
from nvalchemi.dynamics.base import ConvergenceHook
from nvalchemi.dynamics.hooks import FreezeAtomsHook, LoggingHook
from nvalchemi.models.demo import DemoModel, DemoModelWrapper

OUTPUT_DIR = Path("03_ase_integration_output")
OUTPUT_DIR.mkdir(exist_ok=True)

In [5]:
torch.manual_seed(0)
model = DemoModelWrapper(DemoModel())
model.eval()

DemoModelWrapper(
  outputs={energy, forces}
  autograd_outputs={forces}
  (model): DemoModel()
)

In [6]:
def atoms_to_data(atoms) -> AtomicData:
    """Convert an ASE Atoms object to AtomicData with dynamics fields."""
    data = AtomicData.from_atoms(atoms)
    n = data.num_nodes
    data.forces = torch.zeros(n, 3)
    data.energy = torch.zeros(1, 1)
    data.add_node_property("velocities", torch.zeros(n, 3))
    return data


def data_to_atoms(data: AtomicData) -> Atoms:
    """Convert AtomicData back to ASE Atoms for visualization / I/O."""
    atoms = Atoms(
        numbers=data.atomic_numbers.cpu().numpy(),
        positions=data.positions.detach().cpu().numpy(),
    )
    if data.cell is not None:
        atoms.cell = data.cell.squeeze(0).detach().cpu().numpy()
    if data.pbc is not None:
        atoms.pbc = data.pbc.squeeze(0).cpu().numpy()
    return atoms


def batch_to_atoms_list(batch: Batch) -> list[Atoms]:
    """Convert every graph in a Batch to a list of ASE Atoms."""
    return [data_to_atoms(d) for d in batch.to_data_list()]

In [7]:
!ls -ltr

total 8
drwxr-xr-x 1 root root 4096 May 12 13:35 sample_data
drwxr-xr-x 2 root root 4096 May 16 22:52 03_ase_integration_output


In [8]:
print("=== Part 1: FIRE Optimization — Rattled Molecules ===")

molecules = []
for name, seed in [("H2O", 1), ("CH4", 2), ("CH3CH2OH", 3)]:
    mol = molecule(name)
    mol.rattle(stdev=0.15, seed=seed)
    mol.center(vacuum=5.0)
    molecules.append(mol)
    print(f"  {name}: {len(mol)} atoms, rattled")

write(OUTPUT_DIR / "molecules_initial.xyz", molecules)
print(
    f"  Wrote {len(molecules)} initial structures -> {OUTPUT_DIR}/molecules_initial.xyz"
)

data_list_opt = [atoms_to_data(mol) for mol in molecules]
batch_opt = Batch.from_data_list(data_list_opt)
print(f"\nBatch: {batch_opt.num_graphs} systems, {batch_opt.num_nodes} atoms total\n")

fire_opt = FIRE(
    model=model,
    dt=0.1,
    n_steps=200,
    convergence_hook=ConvergenceHook(
        criteria=[
            {"key": "forces", "threshold": 0.05, "reduce_op": "norm", "reduce_dims": -1}
        ]
    ),
)

with LoggingHook(backend="csv", log_path="03_fire_opt.csv", frequency=10) as log_hook:
    fire_opt.register_hook(log_hook)
    batch_opt = fire_opt.run(batch_opt)

print(f"\nCompleted {fire_opt.step_count} FIRE steps. Log: 03_fire_opt.csv")

relaxed_molecules = batch_to_atoms_list(batch_opt)
write(OUTPUT_DIR / "molecules_relaxed.xyz", relaxed_molecules)
print(
    f"Wrote {len(relaxed_molecules)} relaxed structures -> {OUTPUT_DIR}/molecules_relaxed.xyz"
)

=== Part 1: FIRE Optimization — Rattled Molecules ===
  H2O: 3 atoms, rattled
  CH4: 5 atoms, rattled
  CH3CH2OH: 9 atoms, rattled
  Wrote 3 initial structures -> 03_ase_integration_output/molecules_initial.xyz

Batch: 3 systems, 17 atoms total


Completed 1 FIRE steps. Log: 03_fire_opt.csv
Wrote 3 relaxed structures -> 03_ase_integration_output/molecules_relaxed.xyz


In [9]:
print("\n\n=== Part 2: FusedStage — Cu(111) + CO Adsorbate ===")

slab_base = fcc111("Cu", size=(2, 2, 3), vacuum=10.0)

co = molecule("CO")

adsorbate_systems = []
for seed in [10, 11, 12]:
    slab = slab_base.copy()

    # Place CO above the top Cu layer (on-top site)
    top_z = slab.positions[:, 2].max()
    co_copy = co.copy()
    co_copy.translate([slab.cell[0, 0] / 2, slab.cell[1, 1] / 3, top_z + 1.8])

    system = slab + co_copy  # combine slab and adsorbate
    system.rattle(stdev=0.05, seed=seed)
    adsorbate_systems.append(system)
    print(
        f"  System (seed={seed}): {len(system)} atoms "
        f"({len(slab)} slab + {len(co)} adsorbate)"
    )

write(OUTPUT_DIR / "cu111_co_initial.xyz", adsorbate_systems)
print(
    f"  Wrote {len(adsorbate_systems)} initial slab+CO structures "
    f"-> {OUTPUT_DIR}/cu111_co_initial.xyz"
)



=== Part 2: FusedStage — Cu(111) + CO Adsorbate ===
  System (seed=10): 14 atoms (12 slab + 2 adsorbate)
  System (seed=11): 14 atoms (12 slab + 2 adsorbate)
  System (seed=12): 14 atoms (12 slab + 2 adsorbate)
  Wrote 3 initial slab+CO structures -> 03_ase_integration_output/cu111_co_initial.xyz


/usr/local/lib/python3.12/dist-packages/ase/io/extxyz.py:320: UserWarning: Skipping unhashable information adsorbate_info
  warnings.warn('Skipping unhashable information '


In [10]:
data_list_fused = []
for sys in adsorbate_systems:
    data = atoms_to_data(sys)
    tags = torch.tensor(sys.get_tags())
    data.atom_categories = torch.where(
        tags > 0, AtomCategory.SPECIAL.value, AtomCategory.GAS.value
    )
    data_list_fused.append(data)

batch_fused = Batch.from_data_list(data_list_fused)

n_frozen = int((batch_fused.atom_categories == AtomCategory.SPECIAL.value).sum().item())
n_free = int((batch_fused.atom_categories == AtomCategory.GAS.value).sum().item())
print(f"  Frozen (slab): {n_frozen} atoms, Free (adsorbate): {n_free} atoms")

# All systems start in the FIRE stage (status = 0).
batch_fused["status"] = torch.zeros(batch_fused.num_graphs, 1, dtype=torch.long)

print(
    f"\nBatch: {batch_fused.num_graphs} systems, {batch_fused.num_nodes} atoms total\n"
)

  Frozen (slab): 36 atoms, Free (adsorbate): 6 atoms

Batch: 3 systems, 42 atoms total



In [11]:
freeze_hook = FreezeAtomsHook()

# Create LoggingHooks before the stage constructors so they can be passed via
# ``hooks=`` and used as context managers around the run.
fire_logger = LoggingHook(backend="csv", log_path="03_fire_stage.csv", frequency=10)
langevin_logger = LoggingHook(
    backend="csv", log_path="03_langevin_stage.csv", frequency=10
)

In [12]:
#FIRE sub-stage (relaxation) — only adsorbate atoms relax
fire_stage = FIRE(
    model=model,
    dt=0.1,
    convergence_hook=ConvergenceHook(
        criteria=[
            {"key": "forces", "threshold": 0.05, "reduce_op": "norm", "reduce_dims": -1}
        ]
    ),
    hooks=[freeze_hook, fire_logger],
    n_steps=200,
)

# NVT Langevin sub-stage (MD at 300 K) — slab remains frozen
langevin_stage = NVTLangevin(
    model=model,
    dt=0.5,
    temperature=300.0,
    friction=0.1,
    random_seed=42,
    hooks=[freeze_hook, langevin_logger],
    n_steps=200,
)

In [13]:
# Compose: status 0 → FIRE, status 1 → Langevin
fused = fire_stage + langevin_stage
print(f"Created: {fused}\n")

n_fused_steps = 450
with fire_logger, langevin_logger:
    batch_fused = fused.run(batch_fused, n_steps=n_fused_steps)

status_final = batch_fused.status.squeeze(-1).tolist()
print(f"\nFinal status: {status_final}  (0=FIRE, 1=Langevin)")
print(f"FusedStage total steps: {fused.step_count}")

final_systems = batch_to_atoms_list(batch_fused)
write(OUTPUT_DIR / "cu111_co_final.xyz", final_systems)
print(f"Wrote {len(final_systems)} final structures -> {OUTPUT_DIR}/cu111_co_final.xyz")

print(f"\nAll output written to {OUTPUT_DIR.resolve()}/")
print("  Visualize with: ase gui 03_ase_integration_output/cu111_co_initial.xyz")

Created: FusedStage(sub_stages=[0:FIRE, 1:NVTLangevin], entry_status=0, exit_status=2, compiled=False, step_count=0)


Final status: [2, 2, 2]  (0=FIRE, 1=Langevin)
FusedStage total steps: 200
Wrote 3 final structures -> 03_ase_integration_output/cu111_co_final.xyz

All output written to /content/03_ase_integration_output/
  Visualize with: ase gui 03_ase_integration_output/cu111_co_initial.xyz


In [15]:
!ls -ltr 03_ase_integration_output/

total 16
-rw-r--r-- 1 root root 1324 May 16 22:58 molecules_initial.xyz
-rw-r--r-- 1 root root 1053 May 16 22:58 molecules_relaxed.xyz
-rw-r--r-- 1 root root 3123 May 16 22:59 cu111_co_initial.xyz
-rw-r--r-- 1 root root 2718 May 16 23:01 cu111_co_final.xyz


In [16]:
!cp 03_ase_integration_output/cu111_co_final.xyz .

In [17]:
!ls -ltr

total 32
drwxr-xr-x 1 root root 4096 May 12 13:35 sample_data
-rw-r--r-- 1 root root  212 May 16 22:58 03_fire_opt.csv
-rw-r--r-- 1 root root 4369 May 16 23:01 03_fire_stage.csv
-rw-r--r-- 1 root root 4369 May 16 23:01 03_langevin_stage.csv
drwxr-xr-x 2 root root 4096 May 16 23:01 03_ase_integration_output
-rw-r--r-- 1 root root 2718 May 16 23:03 cu111_co_final.xyz


In [20]:
!ase gui ./cu111_co_final.xyz

usage: ase [-h] [--version] [-T]
           {help,info,test,gui,db,run,band-structure,build,dimensionality,eos,ulm,find,nebplot,convert,reciprocal,completion,diff,exec}
           ...
ase: error: TclError: no display name and no $DISPLAY environment variable
To get a full traceback, use: ase -T gui ...


In [1]:
pip install nglview

In [4]:
import nglview as nv
from ase.io import read

# Read the final structures
final_systems = read('cu111_co_final.xyz', index=':')

# Create a list of NGLView components from the ASE Atoms objects
view = nv.show_asetraj(final_systems)
view.add_licorice()
view.center()
view

NGLWidget(max_frame=2)

In [3]:
from google.colab import output
output.enable_custom_widget_manager()

Support for third party widgets will remain active for the duration of the session. To disable support:

In [ ]:
from google.colab import output
output.disable_custom_widget_manager()